# microGPT Benchmark: Python vs Rust

Benchmarks Karpathy's microgpt.py against a 208-line Rust port.
Both train a tiny GPT (4200 params) on a names dataset for 1000 steps.

In [ ]:
# Step 1: Setup - download source files and install Rust
import subprocess, sys, os, time, statistics

# Download microgpt.py from Karpathy's gist
!curl -sL -o microgpt.py https://gist.githubusercontent.com/karpathy/8627fe009c40f57531cb18360106ce95/raw/microgpt.py

# Download microgpt-rust.rs from our gist
!curl -sL -o microgpt-rust.rs https://gist.githubusercontent.com/vinodsharma/64f9460d7c9f2ef4dbfe45591c7a6a6e/raw/microgpt-rust.rs

# Install Rust
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y 2>&1 | tail -3

# Compile Rust version with optimizations
!$HOME/.cargo/bin/rustc -O microgpt-rust.rs -o microgpt-rust 2>&1

print(f"\nPython version: {sys.version}")
print(f"Rust compiler:", end=" ")
!$HOME/.cargo/bin/rustc --version
print("Files ready:", os.path.exists('microgpt.py'), os.path.exists('microgpt-rust'))

In [ ]:
# Step 2: Verify both versions work (single run each)
print("=== Python test run ===")
!python3 microgpt.py 2>&1 | tail -5
print("\n=== Rust test run ===")
!./microgpt-rust 2>&1 | tail -5

In [ ]:
# Step 3: Benchmark - 10 runs each
NUM_RUNS = 10

def bench(cmd, label, num_runs=NUM_RUNS):
    times = []
    for i in range(num_runs):
        start = time.perf_counter()
        result = subprocess.run(cmd, shell=True, capture_output=True)
        elapsed = time.perf_counter() - start
        times.append(elapsed)
        print(f"  {label} run {i+1}/{num_runs}: {elapsed:.2f}s")
    return times

print(f"Running {NUM_RUNS} iterations each...\n")

print("--- Python ---")
py_times = bench("python3 microgpt.py", "Python")

print("\n--- Rust ---")
rs_times = bench("./microgpt-rust", "Rust")

In [ ]:
# Step 4: Results summary
def summary(times):
    return {
        'median': statistics.median(times),
        'mean': statistics.mean(times),
        'min': min(times),
        'max': max(times),
        'stdev': statistics.stdev(times) if len(times) > 1 else 0
    }

py = summary(py_times)
rs = summary(rs_times)
speedup_median = py['median'] / rs['median']
speedup_mean = py['mean'] / rs['mean']

print("=" * 60)
print("BENCHMARK RESULTS")
print("=" * 60)
print(f"{'':20s} {'Python':>12s} {'Rust':>12s}")
print("-" * 60)
print(f"{'Median':20s} {py['median']:>11.2f}s {rs['median']:>11.3f}s")
print(f"{'Mean':20s} {py['mean']:>11.2f}s {rs['mean']:>11.3f}s")
print(f"{'Min':20s} {py['min']:>11.2f}s {rs['min']:>11.3f}s")
print(f"{'Max':20s} {py['max']:>11.2f}s {rs['max']:>11.3f}s")
print(f"{'Std Dev':20s} {py['stdev']:>11.2f}s {rs['stdev']:>11.3f}s")
print("-" * 60)
print(f"{'Speedup (median)':20s} {speedup_median:>11.0f}x")
print(f"{'Speedup (mean)':20s} {speedup_mean:>11.0f}x")
print("=" * 60)
print(f"\nRuns: {NUM_RUNS} | Python: {sys.version.split()[0]}")
!echo -n "Rust: " && $HOME/.cargo/bin/rustc --version
!echo -n "CPU: " && cat /proc/cpuinfo | grep 'model name' | head -1 | cut -d: -f2 | xargs
!echo -n "RAM: " && free -h | grep Mem | awk '{print $2}'
!echo -n "OS: " && cat /etc/os-release | grep PRETTY_NAME | cut -d= -f2 | tr -d '"'